# 01 — Data quality and journey reconstruction

**Goal (Day 1 of the roadmap):** before building labels, check how trustworthy the actual data is, and how to rebuild each train's journey.

1. Dataset overview
2. Defining the scope: IC / IR / RE / EC
3. Swiss vs foreign stops
4. Measured times (`REAL`) by category, operator, station and day
5. Cancellations and extra trips
6. Delay sanity checks
7. Journey reconstruction
8. Prediction points preview
9. Findings and decisions

All heavy lifting is done in DuckDB over `data/interim/trains_*.parquet`; only aggregated tables reach pandas and Plotly.

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from swissdelay import config

pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:,.3f}".format)

# Chart style: fixed categorical order, recessive grid (see the dataviz guidelines)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY = "#8a8984", "#d4d3cd"
pio.templates["swissdelay"] = go.layout.Template(
    layout=dict(
        colorway=SERIES,
        font=dict(family="Inter, system-ui, sans-serif", size=13, color="#0b0b0b"),
        paper_bgcolor="#fcfcfb",
        plot_bgcolor="#fcfcfb",
        xaxis=dict(gridcolor="#e8e7e2", zeroline=False, linecolor="#c3c2b7"),
        yaxis=dict(gridcolor="#e8e7e2", zeroline=False, linecolor="#c3c2b7"),
        bargap=0.25,
        margin=dict(l=60, r=30, t=60, b=50),
    )
)
pio.templates.default = "plotly_white+swissdelay"

SCOPE_CATEGORIES = ("IC", "IR", "RE", "EC")
CAT_COLORS = dict(zip(SCOPE_CATEGORIES, SERIES))
STATUS_COLORS = {"REAL": SERIES[0], "PROGNOSE": SERIES[1], "GESCHAETZT": SERIES[3],
                 "UNBEKANNT": GRAY, "(empty)": LIGHT_GRAY}

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE VIEW trains AS
SELECT
  *,
  operating_day::VARCHAR || '|' || trip_id                 AS trip_key,
  station_id // 100000 = 85                                AS is_swiss,
  upper(regexp_extract(line_name, '^[A-Za-z]+'))           AS line_prefix,
  date_diff('second', arr_planned, arr_actual) / 60.0      AS arr_delay_min,
  date_diff('second', dep_planned, dep_actual) / 60.0      AS dep_delay_min
FROM read_parquet('{config.INTERIM}/trains_*.parquet')
""")
SCOPE = f"category IN {SCOPE_CATEGORIES}"


def q(sql: str) -> pd.DataFrame:
    """Run SQL in DuckDB and return a (small) pandas DataFrame."""
    return con.sql(sql).df()

## 1. Dataset overview

In [ ]:
overview = q("""
SELECT strftime(operating_day, '%Y-%m') AS month,
       count(DISTINCT operating_day)    AS days,
       count(*)                         AS rows,
       count(DISTINCT trip_key)         AS trips,
       count(DISTINCT station_id)       AS stations
FROM trains GROUP BY ALL ORDER BY month
""")
overview

## 2. Defining the scope: IC / IR / RE / EC

`category` is the official train category, but we saw generic values (`Zug`, `ZUG`). Compare it with the prefix of `line_name` (e.g. `IC5` → `IC`) to see whether some in-scope trains hide under another category.

In [ ]:
categories = q("""
SELECT category, count(*) AS rows, count(DISTINCT trip_key) AS trips,
       round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct_rows
FROM trains GROUP BY ALL ORDER BY rows DESC
""")
categories.head(25)

In [ ]:
crosstab = q(f"""
SELECT category, line_prefix, count(*) AS rows, count(DISTINCT trip_key) AS trips
FROM trains
WHERE category IN {SCOPE_CATEGORIES} OR line_prefix IN {SCOPE_CATEGORIES}
GROUP BY ALL ORDER BY rows DESC
""")
# Rows where category and line prefix disagree are the ones to look at
crosstab[crosstab["category"] != crosstab["line_prefix"]].head(30)

In [ ]:
fig = px.bar(
    categories[categories["category"].isin(SCOPE_CATEGORIES)],
    x="category", y="trips", color="category", color_discrete_map=CAT_COLORS,
    title="Train runs in scope, Aug 2025 – Sep 2026", text_auto=".3s",
)
fig.update_layout(showlegend=False, yaxis_title="train runs", xaxis_title=None)
fig.show()

## 3. Swiss vs foreign stops

Swiss stations have a `station_id` starting with 85. A trip is *domestic* (all stops in Switzerland), *cross-border*, or *foreign only* (e.g. the DB RE7 around Augsburg).

In [ ]:
geo = q(f"""
WITH t AS (
  SELECT trip_key, any_value(category) AS category,
         bool_and(is_swiss) AS all_ch, bool_or(is_swiss) AS any_ch
  FROM trains WHERE {SCOPE} GROUP BY trip_key
)
SELECT category,
       CASE WHEN all_ch THEN 'domestic' WHEN any_ch THEN 'cross-border' ELSE 'foreign only' END AS kind,
       count(*) AS trips
FROM t GROUP BY ALL ORDER BY category, kind
""")
geo_wide = geo.pivot(index="category", columns="kind", values="trips").fillna(0)
geo_wide.div(geo_wide.sum(axis=1), axis=0).mul(100).round(1)

In [ ]:
fig = px.bar(
    geo, x="category", y="trips", color="kind", barmode="stack",
    color_discrete_map={"domestic": SERIES[0], "cross-border": SERIES[1], "foreign only": GRAY},
    category_orders={"kind": ["domestic", "cross-border", "foreign only"]},
    title="Where in-scope trains run",
)
fig.update_layout(yaxis_title="train runs", xaxis_title=None, legend_title=None)
fig.show()

In [ ]:
# Operators of foreign-only trips: candidates for exclusion
q(f"""
WITH t AS (SELECT trip_key, any_value(operator_abbr) AS operator, any_value(category) AS category,
                  bool_or(is_swiss) AS any_ch
           FROM trains WHERE {SCOPE} GROUP BY trip_key)
SELECT operator, category, count(*) AS trips FROM t WHERE NOT any_ch
GROUP BY ALL ORDER BY trips DESC LIMIT 15
""")

## 4. Measured times (`REAL`)

Only `REAL` times count as observations or labels. Shares are computed on stops that **have** a planned time (the first stop has no arrival, the last no departure), Swiss stops only.

In [ ]:
status = q(f"""
SELECT category, 'departure' AS event, coalesce(dep_status, '(empty)') AS status, count(*) AS n
FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL GROUP BY ALL
UNION ALL
SELECT category, 'arrival', coalesce(arr_status, '(empty)'), count(*)
FROM trains WHERE {SCOPE} AND is_swiss AND arr_planned IS NOT NULL GROUP BY ALL
""")
status["share"] = status["n"] / status.groupby(["category", "event"])["n"].transform("sum")
status.pivot_table(index=["category", "event"], columns="status", values="share").mul(100).round(1)

In [ ]:
fig = px.bar(
    status, x="share", y="category", color="status", facet_col="event", orientation="h",
    color_discrete_map=STATUS_COLORS,
    category_orders={"status": list(STATUS_COLORS), "category": list(SCOPE_CATEGORIES)},
    title="Status of actual times, Swiss stops", hover_data={"n": ":,"},
)
fig.update_xaxes(tickformat=".0%", title=None)
fig.update_layout(legend_title=None, yaxis_title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
by_operator = q(f"""
SELECT operator_abbr, any_value(operator_name) AS operator_name,
       count(*) AS dep_events,
       avg((dep_status = 'REAL')::INT) AS share_real
FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL
GROUP BY operator_abbr ORDER BY dep_events DESC
""")
by_operator.head(20)

In [ ]:
by_station = q(f"""
SELECT station_id, any_value(station_name) AS station_name,
       count(*) AS dep_events,
       avg((dep_status = 'REAL')::INT) AS share_real
FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL
GROUP BY station_id ORDER BY dep_events DESC
""")
print(f"{len(by_station)} Swiss stations served by IC/IR/RE/EC")
print("Stations with < 50 % REAL departures:", (by_station["share_real"] < 0.5).sum())
by_station.head(20)

In [ ]:
bins = q(f"""
WITH s AS (
  SELECT station_id, avg((dep_status = 'REAL')::INT) AS share_real
  FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL
  GROUP BY station_id HAVING count(*) >= 1000
)
SELECT floor(share_real * 20) / 20 AS bin, count(*) AS stations FROM s GROUP BY ALL ORDER BY bin
""")
fig = px.bar(bins, x="bin", y="stations", title="Share of REAL departures per station (≥ 1,000 departures)")
fig.update_traces(marker_color=SERIES[0])
fig.update_layout(xaxis_title="share REAL (5 % bins)", yaxis_title="stations", xaxis_tickformat=".0%")
fig.show()

In [ ]:
daily_real = q(f"""
SELECT operating_day, category, avg((dep_status = 'REAL')::INT) AS share_real, count(*) AS dep_events
FROM trains WHERE {SCOPE} AND is_swiss AND dep_planned IS NOT NULL
GROUP BY ALL ORDER BY operating_day
""")
fig = px.line(daily_real, x="operating_day", y="share_real", color="category",
              color_discrete_map=CAT_COLORS, title="Daily share of REAL departures")
fig.update_layout(yaxis_tickformat=".0%", xaxis_title=None, yaxis_title=None, legend_title=None,
                  hovermode="x unified")
fig.show()

# Days where measurement dropped (possible feed outages)
d = daily_real.groupby("operating_day").apply(
    lambda g: (g["share_real"] * g["dep_events"]).sum() / g["dep_events"].sum()
).rename("share_real").reset_index()
thr = d["share_real"].mean() - 3 * d["share_real"].std()
d[d["share_real"] < thr]

## 5. Cancellations and extra trips

A trip is *fully cancelled* if every stop is cancelled, *partially cancelled* if only some are (e.g. a train turned back early). Days with many cancellations are candidates for the **disruption days** breakdown.

In [ ]:
cancel_daily = q(f"""
WITH t AS (
  SELECT operating_day, trip_key, bool_and(is_cancelled) AS full_c, bool_or(is_cancelled) AS any_c,
         bool_or(is_extra_trip) AS extra
  FROM trains WHERE {SCOPE} GROUP BY ALL
)
SELECT operating_day, count(*) AS trips,
       avg(full_c::INT) AS fully_cancelled,
       avg((any_c AND NOT full_c)::INT) AS partially_cancelled,
       avg(extra::INT) AS extra_trips
FROM t GROUP BY ALL ORDER BY operating_day
""")
cancel_daily[["fully_cancelled", "partially_cancelled", "extra_trips"]].describe().mul(100).round(2)

In [ ]:
long = cancel_daily.melt(id_vars="operating_day",
                         value_vars=["fully_cancelled", "partially_cancelled"],
                         var_name="type", value_name="share")
fig = px.line(long, x="operating_day", y="share", color="type",
              color_discrete_map={"fully_cancelled": SERIES[7], "partially_cancelled": SERIES[1]},
              title="Share of in-scope train runs cancelled, per day")
fig.update_layout(yaxis_tickformat=".1%", xaxis_title=None, yaxis_title=None, legend_title=None,
                  hovermode="x unified")
fig.show()
cancel_daily.nlargest(10, "fully_cancelled")

## 6. Delay sanity checks

Departure delay, REAL only, Swiss stops. Planned times are rounded to the minute, so small values (< 30 s) are partly rounding noise.

In [ ]:
q(f"""
SELECT category,
       count(*) AS n,
       avg(dep_delay_min) AS mean,
       quantile_cont(dep_delay_min, [0.01, 0.1, 0.5, 0.9, 0.99]) AS q01_q10_q50_q90_q99,
       avg((dep_delay_min < -1)::INT)  AS share_early_1min,
       avg((dep_delay_min > 3)::INT)   AS share_over_3min,
       avg((dep_delay_min > 60)::INT)  AS share_over_60min,
       max(dep_delay_min) AS max
FROM trains WHERE {SCOPE} AND is_swiss AND dep_status = 'REAL'
GROUP BY ALL ORDER BY category
""")

In [ ]:
hist = q(f"""
SELECT category, greatest(least(floor(dep_delay_min), 20), -3) AS minute, count(*) AS n
FROM trains WHERE {SCOPE} AND is_swiss AND dep_status = 'REAL'
GROUP BY ALL ORDER BY minute
""")
hist["share"] = hist["n"] / hist.groupby("category")["n"].transform("sum")
fig = px.bar(hist, x="minute", y="share", color="category", barmode="group",
             color_discrete_map=CAT_COLORS, category_orders={"category": list(SCOPE_CATEGORIES)},
             title="Departure delay distribution (clipped to [-3, 20] min)")
fig.update_layout(yaxis_tickformat=".0%", xaxis_title="delay (min, floored)", yaxis_title=None,
                  legend_title=None)
fig.show()

In [ ]:
# Physically impossible or suspicious combinations
q(f"""
SELECT
  count(*) FILTER (WHERE arr_status = 'REAL' AND dep_status = 'REAL' AND dep_actual < arr_actual)
    AS dep_before_arr,
  count(*) FILTER (WHERE arr_planned > dep_planned)                AS planned_dep_before_arr,
  count(*) FILTER (WHERE dep_status = 'REAL' AND dep_delay_min < -5) AS early_over_5min,
  count(*) FILTER (WHERE dep_status = 'REAL' AND dep_delay_min > 180) AS late_over_3h,
  count(*) FILTER (WHERE is_pass_through)                           AS pass_through_rows,
  count(*) FILTER (WHERE is_pass_through AND dep_status = 'REAL')   AS pass_through_with_real
FROM trains WHERE {SCOPE}
""").T

## 7. Journey reconstruction

One journey = one `(operating_day, trip_id)`. Stops are ordered by planned time; the checks below tell us whether that ordering is reliable and whether `trip_key` really identifies a single train run.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE journeys AS
SELECT *,
  row_number() OVER w AS stop_seq,
  count(*) OVER (PARTITION BY trip_key) AS n_stops,
  lead(arr_planned) OVER w AS next_arr_planned,
  lead(station_id)  OVER w AS next_station_id
FROM trains
WHERE {SCOPE}
WINDOW w AS (PARTITION BY trip_key
             ORDER BY coalesce(arr_planned, dep_planned), coalesce(dep_planned, arr_planned), station_id)
""")
q("SELECT count(*) AS rows, count(DISTINCT trip_key) AS journeys FROM journeys")

In [ ]:
q("""
WITH j AS (
  SELECT trip_key,
         count(*) AS n_stops,
         count(DISTINCT station_id) AS n_stations,
         count(DISTINCT operator_id) AS n_operators,
         count(DISTINCT category) AS n_categories,
         count(DISTINCT train_number) AS n_train_numbers,
         bool_or(stop_seq = 1 AND arr_planned IS NOT NULL) AS first_has_arr,
         bool_or(stop_seq = n_stops AND dep_planned IS NOT NULL) AS last_has_dep,
         count(*) FILTER (WHERE dep_planned > next_arr_planned) AS order_violations,
         count(*) FILTER (WHERE station_id = next_station_id) AS consecutive_duplicates
  FROM journeys GROUP BY trip_key
)
SELECT
  count(*)                                   AS journeys,
  avg((n_stops = 1)::INT)                    AS share_single_stop,
  avg((n_stations < n_stops)::INT)           AS share_repeated_station,
  avg((n_operators > 1)::INT)                AS share_multi_operator,
  avg((n_categories > 1)::INT)               AS share_multi_category,
  avg((n_train_numbers > 1)::INT)            AS share_multi_train_number,
  avg(first_has_arr::INT)                    AS share_first_stop_has_arr,
  avg(last_has_dep::INT)                     AS share_last_stop_has_dep,
  avg((order_violations > 0)::INT)           AS share_order_violation,
  avg((consecutive_duplicates > 0)::INT)     AS share_consecutive_duplicate
FROM j
""").T

In [ ]:
stops = q("""
SELECT category, least(n_stops, 40) AS n_stops, count(DISTINCT trip_key) AS journeys
FROM journeys GROUP BY ALL ORDER BY n_stops
""")
fig = px.bar(stops, x="n_stops", y="journeys", color="category", barmode="stack",
             color_discrete_map=CAT_COLORS, category_orders={"category": list(SCOPE_CATEGORIES)},
             title="Stops per journey (40 = 40 or more)")
fig.update_layout(xaxis_title="stops", yaxis_title="journeys", legend_title=None)
fig.show()

In [ ]:
# Journeys running past midnight: planned times on the next calendar day
q("""
SELECT avg((max_t::DATE > operating_day)::INT) AS share_past_midnight
FROM (SELECT trip_key, any_value(operating_day) AS operating_day,
             max(coalesce(dep_planned, arr_planned)) AS max_t
      FROM journeys GROUP BY trip_key)
""")

In [ ]:
# How delay evolves along a journey: mean REAL departure delay by stop number
along = q("""
SELECT category, stop_seq, avg(dep_delay_min) AS mean_delay, count(*) AS n
FROM journeys
WHERE is_swiss AND dep_status = 'REAL' AND stop_seq <= 25
GROUP BY ALL ORDER BY stop_seq
""")
fig = px.line(along, x="stop_seq", y="mean_delay", color="category", markers=True,
              color_discrete_map=CAT_COLORS, category_orders={"category": list(SCOPE_CATEGORIES)},
              title="Mean departure delay along the journey", hover_data={"n": ":,"})
fig.update_layout(xaxis_title="stop number", yaxis_title="mean delay (min)", legend_title=None,
                  hovermode="x unified")
fig.show()

In [ ]:
# One example journey: a long IC run on a busy weekday
example = q("""
SELECT trip_key FROM journeys
WHERE category = 'IC' AND is_swiss AND n_stops >= 10 AND operating_day = DATE '2026-03-10'
GROUP BY trip_key ORDER BY trip_key LIMIT 1
""")["trip_key"][0]
trip = q(f"""
SELECT stop_seq, station_name, line_name, arr_planned, arr_actual, arr_status, arr_delay_min,
       dep_planned, dep_actual, dep_status, dep_delay_min, is_cancelled
FROM journeys WHERE trip_key = '{example}' ORDER BY stop_seq
""")
print(example)
trip

In [ ]:
pts = pd.concat([
    trip[["station_name", "arr_planned", "arr_delay_min", "arr_status"]]
        .rename(columns={"arr_planned": "planned", "arr_delay_min": "delay", "arr_status": "status"})
        .assign(event="arrival"),
    trip[["station_name", "dep_planned", "dep_delay_min", "dep_status"]]
        .rename(columns={"dep_planned": "planned", "dep_delay_min": "delay", "dep_status": "status"})
        .assign(event="departure"),
]).dropna(subset=["planned"]).sort_values("planned")
fig = px.line(pts, x="planned", y="delay", markers=True, hover_data=["station_name", "event", "status"],
              title=f"Delay along one IC journey ({example})")
fig.update_traces(line_color=SERIES[0])
fig.update_layout(xaxis_title="planned time", yaxis_title="delay (min)")
fig.show()

## 8. Prediction points preview

Prediction points are measured (`REAL`) departures of in-scope trains. This is the upper bound on the training set before horizon labels are built (Day 2).

In [ ]:
pp = q(f"""
SELECT strftime(operating_day, '%Y-%m') AS month,
       count(*) FILTER (WHERE dep_planned IS NOT NULL)                  AS departures,
       count(*) FILTER (WHERE dep_status = 'REAL' AND NOT is_cancelled) AS prediction_points
FROM trains WHERE {SCOPE} AND is_swiss
GROUP BY ALL ORDER BY month
""")
pp["share"] = pp["prediction_points"] / pp["departures"]
pp

## 9. Findings and decisions

*To fill in after running the notebook.*

| Question | Finding | Decision |
|---|---|---|
| Scope: is `category` enough, or also `line_name`? | | |
| Foreign-only trips | | |
| Foreign stops of cross-border trips | | |
| Operators / stations with low `REAL` share | | |
| Feed outage days | | |
| Disruption-day definition (cancellations) | | |
| Pass-through rows | | |
| Journey key and stop ordering | | |
| Journeys past midnight | | |